# 1차시 — 무엇을 예측할 것인가

### MAPS 기반 다문화청소년 문화적응 스트레스 머신러닝 · 문제 정의

> **오늘 한 문장:** "우리는 *중2 때의 심리사회적 환경*으로 *중3 때의 높은 문화적응 스트레스*를
> 구분해 보려 한다. 오늘은 코드가 아니라 **무엇을 예측할지**를 확정한다."

이 프로젝트는 다문화청소년패널조사(MAPS) 1기 자료로
**5차년도(2015, 중2) → 6차년도(2016, 중3)** 의 1년 예측 문제를 푼다.

오늘의 목표 4가지:

1. **문화적응 스트레스**가 무엇이고 왜 이걸 재는지 이해한다.
2. **예측(prediction)과 인과(causation)** 의 차이를 말로 설명할 수 있다.
3. 우리 문제를 머신러닝 언어(**feature / target / classification**)로 번역한다.
4. **데이터가 실제로 있는지 확인**하고, 없으면 없다고 정확히 보고한다.

> 💡 **이번 과정 운영 방식:** 셀을 위에서 아래로 하나씩 실행한다.
> 코드는 **전부 채워져 있다.** 셀을 실행하기 **전에** 결과를 예측해 보고,
> 바로 아래 `# CHECK` 셀에서 `✅` 가 뜨는지 확인한 뒤 다음으로 넘어간다.

## 🗺️ 프로젝트 전체 지도 — 8차시

| 차시 | 심리학 | IT / ML |
|---|---|---|
| **1 (오늘)** | 문화적응 스트레스 · 예측 vs 인과 · 연구윤리 | 데이터셋 · feature/target · classification |
| 2 | 심리척도 · 문항 · 역채점 | pandas · 결측치 · ID join |
| 3 | 평균/SD/분포/상관 · Cronbach α | 집계 · 시각화 |
| 4 | 고스트레스 집단의 **조작적 정의** | train/test · 불균형 · **데이터 누출** |
| 5 | 예측변수의 방향성 | 로지스틱 회귀 · 표준화 계수 |
| 6 | 심리는 선형적으로 작동하는가 | Decision Tree · Random Forest · 과적합 |
| 7 | 위험요인 · 보호요인 | Permutation Importance |
| 8 | 결론 · 한계 · 윤리 | 재현성 · 최종 리포트 |

**연구 질문 3개**

- **RQ1** 중2 시점 심리사회적 특성으로 1년 후 고스트레스 집단을 어느 정도 구분할 수 있는가?
- **RQ2** 그 구분에 상대적으로 중요한 변수는 무엇인가?
- **RQ3** 이전 시점의 스트레스를 넣으면 예측력이 얼마나 개선되는가?

## Step 0 — 환경 설정

오늘은 무거운 라이브러리가 필요 없다. 표를 다루는 `pandas` 와 설정을 읽는 `pyyaml` 뿐이다.

In [ ]:
!pip install pandas pyyaml -q

In [ ]:
# ── 프로젝트 환경 자동 설정 (Colab / 로컬 공용) ───────────────────────
# 이 셀은 모든 차시 노트북 맨 위에 동일하게 들어간다. 그냥 실행만 하면 된다.
#
# Colab 사용법: 구글 드라이브('내 드라이브' 하위 2단계까지) 아무 곳에나
#   program5 zip 을 하나 올려 두면 된다. 이 셀이 드라이브를 mount 하고
#   /content 에 압축까지 풀어 준다. 런타임이 끊겨도 이 셀만 다시 실행하면
#   되고, 32MB zip 을 매번 재업로드할 필요가 없다.
import os, sys, glob, zipfile

DRIVE_ZIP_PATTERNS = [
    "/content/drive/MyDrive/program5*.zip",
    "/content/drive/MyDrive/*/program5*.zip",
    "/content/drive/MyDrive/*/*/program5*.zip",
]


# program5 로 인정하려면 이 4개가 다 있어야 한다.
# 왜 4개나 보나: 이름만 program5 인 '반쪽 폴더'(예전에 일부만 풀렸거나 업로드가 끊긴 것)를
#   붙잡으면 한참 뒤 4차시에서 "scripts/build_dataset.py 없음"으로 터진다. 여기서 거른다.
REQUIRED = ["AGENTS.md", "configs/variables.yaml", "configs/modeling.yaml",
            "scripts/build_dataset.py", "src/maps_risk/__init__.py"]


def missing_parts(path):
    """그 폴더에서 REQUIRED 중 빠진 파일 목록. 비어 있으면 온전한 프로젝트다."""
    return [f for f in REQUIRED if not os.path.isfile(os.path.join(path, *f.split("/")))]


def is_project(path):
    return not missing_parts(path)


def find_project():
    """이미 풀려 있는 program5 폴더를 후보 경로에서 찾는다.

    온전한 폴더만 고른다. program5 처럼 보이는데 반쪽인 폴더는 건너뛰되,
    **무엇이 없어서 건너뛰었는지 반드시 출력한다** — 조용히 넘어가면 원인 못 찾는다.
    """
    found, half, seen = None, [], set()
    for c in [".", "program5", "..", "../program5", "/content/program5",
              "/content/edu/program5", os.path.expanduser("~/program5")]:
        if not os.path.isdir(c):
            continue
        real = os.path.realpath(c)          # 같은 폴더를 두 경로로 가리키면 한 번만 본다
        if real in seen:
            continue
        seen.add(real)
        miss = missing_parts(c)
        if not miss:
            found = found or os.path.abspath(c)
        elif os.path.isfile(os.path.join(c, "AGENTS.md")):   # program5 인 척하는 반쪽 폴더
            half.append((os.path.abspath(c), miss))
    for path, miss in half:
        print("⚠️  반쪽 폴더라 건너뛴다:", path)
        print("     없는 것:", ", ".join(miss))
    return found


def in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


def member_name(info):
    """UTF-8 플래그가 없는 zip 의 한글 파일명을 되살린다.

    맥 /usr/bin/zip 은 EFS(0x800) 플래그를 세우지 않는다. 그러면 zipfile 이
    이름을 cp437 로 잘못 디코딩해 'φòÖδ╢Ç…' 같은 폴더가 생기고, 뒤이어
    data/raw 스캔이 0개를 돌려준다. 원래 바이트로 되돌려 다시 읽는다.
    """
    if info.flag_bits & 0x800:          # 이미 UTF-8 로 제대로 읽힌 이름
        return info.filename
    try:
        raw = info.filename.encode("cp437")
    except UnicodeEncodeError:
        return info.filename
    for enc in ("utf-8", "cp949"):      # 한글 zip 은 이 둘 중 하나다
        try:
            return raw.decode(enc)
        except UnicodeDecodeError:
            pass
    return info.filename


def setup_from_drive():
    """드라이브를 mount 하고 program5 zip 을 /content 에 푼다. 경로 또는 None."""
    from google.colab import drive
    drive.mount("/content/drive")   # 이미 붙어 있으면 그대로 통과한다

    # MyDrive 전체를 재귀 탐색하면 느리다 — 하위 2단계까지만 훑는다.
    zips = [z for p in DRIVE_ZIP_PATTERNS for z in sorted(glob.glob(p))]
    if not zips:
        print("⚠️  드라이브에서 program5*.zip 을 찾지 못했다.")
        print("   '내 드라이브' 또는 그 하위 2단계 폴더에 zip 을 두고 이 셀을 다시 실행하라.")
        print("   탐색한 위치:")
        for p in DRIVE_ZIP_PATTERNS:
            print("     ", p)
        return None

    src = zips[0]
    if len(zips) > 1:
        print(f"ℹ️  zip 후보 {len(zips)}개 중 첫 번째를 쓴다:",
              [os.path.basename(z) for z in zips])
    print(f"📦 {os.path.basename(src)} ({os.path.getsize(src) / 1e6:.1f} MB) → /content 에 푸는 중…")
    with zipfile.ZipFile(src) as zf:
        for info in zf.infolist():      # zip 안에 program5/ 폴더가 들어 있다
            info.filename = member_name(info)
            zf.extract(info, "/content")
    return find_project()


PROJECT = find_project()
if PROJECT is None and in_colab():
    PROJECT = setup_from_drive()

if PROJECT is None:
    print("🛑 프로젝트 폴더를 찾지 못했다. 아래 중 하나로 해결한다:")
    print("  (A) Colab: 구글 드라이브 '내 드라이브'에 program5 zip 을 올리고 이 셀 재실행")
    print("  (B) Colab: 좌측 파일창에 zip 을 올린 뒤  !unzip -q -o program5*.zip")
    print("      ※ 위에 '반쪽 폴더' 경고가 떴다면 그 폴더를 지우고 다시 풀어야 한다:")
    print("        !rm -rf /content/program5   ← 그 뒤 이 셀 재실행")
    print("  (C) 로컬 : program5 폴더 안(또는 그 상위)에서 노트북을 열었는지 확인")
    print("\n지금 /content 에 있는 것:", sorted(os.listdir("/content"))[:20]
          if os.path.isdir("/content") else "(없음)")
    # 여기서 멈춘다. 경고만 찍고 넘어가면 cwd 도 sys.path 도 안 잡힌 채로
    # 아래 셀들이 ModuleNotFoundError → FileNotFoundError 로 줄줄이 터진다.
    # sys.path 를 손으로 채워 봐야 cwd 가 여전히 /content 라 configs/*.yaml 을 못 읽는다.
    raise RuntimeError("program5 프로젝트 폴더를 찾지 못했다 — 위 안내대로 조치한 뒤 이 셀을 다시 실행하라.")

os.chdir(PROJECT)
src_dir = os.path.join(PROJECT, "src")
if src_dir not in sys.path:      # 셀을 여러 번 돌려도 중복 추가되지 않게
    sys.path.insert(0, src_dir)
print("✅ 프로젝트 경로:", PROJECT)


In [ ]:
# ── 차시 간 산출물 전달: 구글 드라이브에 저장/복원 ─────────────────────
# Colab 런타임은 끊기면 /content 가 사라진다. 그래서 "다음 차시가 필요로 하는 파일"은
# 내 드라이브에 따로 보관한다 — 그러면 차시 사이에 파일을 손으로 들고 다니지 않아도 된다.
#
#   저장 위치: 내 드라이브/program5_state/   (프로젝트와 같은 경로 구조로 쌓인다)
#     program5_state/configs/variables.yaml
#     program5_state/data/processed/modeling_frame.parquet
#     program5_state/reports/...
#
# 🔴 이 폴더에는 MAPS 원자료에서 파생된 파일이 들어간다. **개인 계정 안에만** 두고
#    링크 공유·양도하지 않는다 (MAPS 이용 조건). 공용 드라이브에 두지 말 것.
import filecmp as _filecmp
import glob as _glob
import os as _os
import shutil as _shutil

STATE_DIR = _os.environ.get("PROGRAM5_STATE_DIR")     # 로컬 테스트용 수동 지정


def _in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


def _state_dir(create=False):
    """전달 폴더 경로. Colab 이 아니고 지정도 없으면 None 이라 그냥 건너뛴다."""
    global STATE_DIR
    if STATE_DIR:
        if create:
            _os.makedirs(STATE_DIR, exist_ok=True)
        return STATE_DIR
    if not _in_colab():
        return None
    from google.colab import drive
    drive.mount("/content/drive")          # 이미 붙어 있으면 그대로 통과한다
    STATE_DIR = "/content/drive/MyDrive/program5_state"
    _os.makedirs(STATE_DIR, exist_ok=True)
    return STATE_DIR


def handoff_push(patterns, label="다음 차시로 넘길 것을 드라이브에 저장"):
    """지금 만든 산출물을 드라이브에 저장한다.

    받는 것: 프로젝트 기준 상대경로 목록 (glob 가능. 예: reports/figures/*.png)
    돌려주는 것: 실제로 저장한 경로 리스트
    왜: 다음 차시가 이 파일을 "없으면 못 여는 재료"로 쓰기 때문이다.
    """
    print("📤 " + label)
    root = _state_dir(create=True)
    if root is None:
        print("   로컬 환경 — 저장을 건너뛴다 (파일이 이미 디스크에 그대로 남는다).")
        return []
    saved = []
    for pat in patterns:
        hits = sorted(_glob.glob(pat))
        if not hits:
            print("   ⬜ " + pat + " — 아직 없다 (이번 차시에서 만들지 않았다면 정상)")
            continue
        for src in hits:
            if not _os.path.isfile(src):
                continue
            dst = _os.path.join(root, src)
            _os.makedirs(_os.path.dirname(dst), exist_ok=True)
            _shutil.copy2(src, dst)
            saved.append(src)
            print("   ✅ " + src + "  →  드라이브")
    print("   저장 위치: " + root)
    return saved


def _yaml_completeness(path):
    """variables.yaml 이 얼마나 채워져 있는지 (게이트 열림, 검증된 구성개념 수).

    왜 필요한가: 드라이브에 **예전의 빈 variables.yaml** 이 남아 있는 경우가 있다.
    그걸 zip 의 검증본 위에 덮어쓰면 build_dataset.py 가 Human Review Gate 에서 멈춘다
    ("codebook_verified 가 false / 문항이 비어 있다"). 파일이 새것인지는 알 수 없어도
    **어느 쪽이 더 채워져 있는지**는 알 수 있다 — 덜 채워진 쪽으로는 덮어쓰지 않는다.
    """
    try:
        import yaml as _yaml
        d = _yaml.safe_load(open(path, encoding="utf-8")) or {}
    except Exception:
        return (0, 0)
    gate = bool((d.get("meta") or {}).get("codebook_verified"))
    n = sum(1 for sec in ("predictors", "optional_predictors")
            for spec in (d.get(sec) or {}).values()
            if spec.get("status") == "verified" and spec.get("items"))
    n += len((d.get("target") or {}).get("items") or [])
    return (int(gate), n)


# 파일별 '퇴보 방지' 검사. 드라이브 사본 점수가 지금 것보다 낮으면 그냥 둔다.
DOWNGRADE_GUARD = {"configs/variables.yaml": _yaml_completeness}


def handoff_pull(patterns, overwrite=True, label="지난 차시 산출물을 드라이브에서 복원"):
    """이번 차시에 필요한 파일을 드라이브에서 가져온다.

    받는 것: 상대경로 목록 (glob 가능), overwrite — 이미 있는 파일도 덮어쓸지 (기본 True)
    돌려주는 것: 실제로 가져온 경로 리스트
    왜 기본이 덮어쓰기인가: zip 안에 **같은 이름의 출발점 파일**이 이미 들어 있다
      (configs/variables.yaml · reports/model_metrics_cv.csv …). '없을 때만' 가져오면
      zip 의 옛 파일이 항상 이겨서 **지난 차시가 고친 내용이 영영 전달되지 않는다.**
      드라이브에 있는 것은 정의상 '지난 차시가 끝내고 밀어 넣은 최신본'이므로 그쪽을 쓴다.
    """
    print("📥 " + label)
    root = _state_dir()
    if root is None:
        print("   로컬 환경 — 복원을 건너뛴다 (디스크의 파일을 그대로 쓴다).")
        return []
    got = []
    for pat in patterns:
        hits = sorted(_glob.glob(_os.path.join(root, pat)))
        if not hits:
            print("   ⬜ " + pat + " — 드라이브에도 없다")
            continue
        for src in hits:
            rel = _os.path.relpath(src, root)
            exists = _os.path.exists(rel)
            if exists and _filecmp.cmp(src, rel, shallow=False):
                print("   ↩︎ " + rel + " — 드라이브와 내용이 같다 (그대로 둔다)")
                continue
            if exists and not overwrite:
                print("   ⚠️ " + rel + " — 드라이브 쪽과 다른데 덮어쓰지 않았다 (overwrite=False)")
                continue
            score = DOWNGRADE_GUARD.get(rel.replace(_os.sep, "/"))
            if exists and score and score(src) < score(rel):
                print("   🛡 " + rel + " — 드라이브 사본이 **더 비어 있다**. 지금 것을 그대로 쓴다.")
                print("       드라이브: " + str(score(src)) + " · 지금: " + str(score(rel))
                      + "   (게이트 열림, 검증된 구성개념 수)")
                print("       드라이브에 옛 파일이 남아 있는 것이다 — 이번 차시 끝에서 새것으로 덮인다.")
                continue
            _os.makedirs(_os.path.dirname(rel) or ".", exist_ok=True)
            _shutil.copy2(src, rel)
            got.append(rel)
            print(("   🔄 " if exists else "   ✅ ") + rel + "  ←  드라이브"
                  + ("  (zip 의 옛 파일을 덮어썼다)" if exists else ""))
    return got


def handoff_require(paths, hint=""):
    """이번 차시의 "없으면 못 여는 재료"를 확인한다. 없으면 이유를 알려준다."""
    missing = [p for p in paths if not _glob.glob(p)]
    if missing:
        print("\n🛑 이번 차시에 꼭 필요한 파일이 없다:")
        for m in missing:
            print("   -", m)
        if hint:
            print("   → " + hint)
        print("   → 지난 차시 노트북을 열어 **맨 끝의 '드라이브에 저장' 셀**을 실행한 뒤 돌아오라.")
    else:
        print("\n✅ 이번 차시에 필요한 재료가 전부 있다.")
    return not missing


handoff_pull([
])

print("1차시는 가져올 것이 없다 (첫 차시). 오늘 만든 것을 **맨 끝 셀에서 드라이브에 저장**한다.")


## Step 1 — 문화적응 스트레스란 무엇인가 🧠

**문화적응(acculturation)** 은 다른 문화와 지속적으로 접촉하면서 일어나는 변화다.
그 과정에서 겪는 심리적 부담이 **문화적응 스트레스(acculturative stress)** 다.

다문화청소년에게 이건 추상적인 개념이 아니다. 구체적으로는 이런 경험이다:

- 한국어가 서툴러 수업을 따라가기 어렵다
- 부모의 출신 국가 때문에 친구들에게 놀림을 받는다
- 집에서 쓰는 문화와 학교에서 요구하는 문화가 다르다
- "나는 한국인인가, 아닌가"라는 정체성 혼란

MAPS는 이걸 **SAFE 척도 10문항 · 4점 리커트**로 잰다
(원척도: Mena, Padilla, & Maldonado, 1987 — Hong, 2003 수정·보완판).
점수가 높을수록 스트레스가 크다.

> 📌 **왜 이 변수를 예측하려 하나?** 문화적응 스트레스는 우울·사회적 위축·학교부적응의
> 선행 요인으로 반복 보고돼 왔다. **1년 뒤**를 미리 구분할 수 있다면, 그 구분에 기여한
> 변수들이 곧 **개입의 후보**가 된다 — 다만 그건 가설이지 증명이 아니다(Step 3에서 다룬다).

## Step 2 — 위험요인과 보호요인

같은 환경에서도 어떤 청소년은 스트레스가 높고 어떤 청소년은 낮다. 그 차이를 만드는 것을
심리학에서는 두 가지로 나눠 본다.

| | 뜻 | 이 프로젝트의 후보 변수 |
|---|---|---|
| **위험요인 (risk factor)** | 있을수록 나쁜 결과 가능성을 높이는 것 | 우울, 사회적 위축, 집단괴롭힘 피해 |
| **보호요인 (protective factor)** | 있을수록 나쁜 결과를 완충하는 것 | 가족지지, 친구지지, 교사지지, 자아탄력성, 이중문화 수용태도 |

우리 모델의 예측변수는 대부분 이 두 범주에 들어간다.
**중요:** 모델은 "위험/보호"를 알지 못한다. 그건 **우리가 붙이는 해석**이다.
7차시에서 계수의 **부호(+/−)** 를 보고 이 분류가 데이터와 맞는지 확인하게 된다.

In [ ]:
# 오늘은 개념을 코드로 한 번 적어 둔다. 2차시에 실제 컬럼과 연결된다.
risk = ["depression", "social_withdrawal", "bullying"]
protective = ["family_support", "peer_support", "teacher_support",
              "ego_resilience", "bicultural_attitude"]

print("위험요인 후보 :", len(risk), "개 →", risk)
print("보호요인 후보 :", len(protective), "개 →", protective)
print("\n※ 이 분류는 '가설'이다. 데이터가 반대로 나올 수도 있고, 그것도 결과다.")

## Step 3 — 예측 vs 인과 (오늘의 첫 번째 봉우리) ⚠️

이 프로젝트에서 **가장 많이 틀리는 지점**이다.

우리 모델이 "우울이 가장 중요한 변수다"라고 말했다고 하자. 이때 할 수 있는 말과 없는 말:

| | 문장 | 왜 |
|---|---|---|
| ❌ | "우울이 문화적응 스트레스를 **일으켰다**" | 인과 주장. 실험도 통제도 하지 않았다 |
| ❌ | "우울을 낮추면 스트레스가 낮아질 것이다" | 개입 효과 주장. 우리 설계로는 알 수 없다 |
| ✅ | "본 데이터와 모델에서 우울 변수가 차년도 고스트레스 집단 **분류에 상대적으로 높은 예측 기여**를 보였다" | 관찰된 연관성만 진술 |

**왜 인과를 말할 수 없나?**

1. **교란변수(confounder)** — 가정 경제 수준이 우울도 높이고 스트레스도 높였을 수 있다.
2. **역방향 가능성** — 이미 스트레스가 높아서 우울해진 것일 수도 있다(5차 스트레스를 Model B에서 넣는 이유).
3. **무작위 배정 없음** — 누가 어떤 환경에 놓일지 우리가 정하지 않았다.

> 시간 순서(2015 → 2016)를 지킨 것은 **역방향을 줄이는 장치**이지 인과의 증명이 아니다.

In [ ]:
# 우리가 결과에 쓸 수 있는 문장은 어느 것인가 — 정답과 이유를 함께 읽는다
statements = {
    "우울이 문화적응 스트레스를 유발한다":            False,   # ❌ "유발한다" = 인과 주장
    "우울 점수가 높은 집단에서 고스트레스 분류 비율이 높았다": True,    # ✅ 관찰된 연관
    "가족지지를 늘리면 스트레스가 줄어든다":            False,   # ❌ "늘리면 줄어든다" = 개입 효과 주장
    "가족지지는 고스트레스 분류에 음(-)의 기여를 보였다":  True,    # ✅ 모델이 실제로 보인 것
}
for s, ok in statements.items():
    print(("✅ 쓸 수 있음  " if ok else "❌ 쓸 수 없음  ") + s)

In [ ]:
# CHECK Step3
try:
    answer = [False, True, False, True]
    got = list(statements.values())
    assert got == answer, f"정답은 {answer} 인데 {got} 로 적었다"
    print("✅ PASS — 예측과 인과를 구분했다. 이 감각을 8차시 내내 유지한다.")
except Exception as e:
    print("❌ FAIL —", e, "\n힌트: '유발한다', '늘리면 줄어든다'는 모두 인과·개입 주장이다.")

<details><summary>💡 해설 (펼쳐 보기)</summary>

```python
statements = {
    "우울이 문화적응 스트레스를 유발한다":            False,
    "우울 점수가 높은 집단에서 고스트레스 분류 비율이 높았다": True,
    "가족지지를 늘리면 스트레스가 줄어든다":            False,
    "가족지지는 고스트레스 분류에 음(-)의 기여를 보였다":  True,
}
```
**"유발한다" · "늘리면 줄어든다"** → 인과·개입 주장이라 쓸 수 없다.
**"비율이 높았다" · "기여를 보였다"** → 관찰된 연관성이라 쓸 수 있다.
</details>

## Step 4 — 우리 문제를 머신러닝 언어로 번역하기

심리학 문장을 ML 용어로 바꾸는 연습이다. 이게 오늘의 핵심 작업이다.

| 심리학 언어 | ML 언어 | 우리 경우 |
|---|---|---|
| 연구 대상 1명 | **row (행)** | 응답자 1명 = 1행 |
| 측정한 특성 하나 | **column (열)** | 우울 점수, 가족지지 점수… |
| 예측에 쓰는 특성 | **feature (X)** | 5차년도(2015) 심리사회 변인들 |
| 맞히려는 것 | **target (y)** | 6차년도(2016) 고스트레스 여부 |
| 집단을 둘로 나누기 | **binary classification** | high_stress = 1 / 0 |

**target 만들기 (4차시에 코드로 구현)**

```
6차 문화적응 스트레스 10문항
  → 역채점 확인 → 문항 평균 → 연속 점수
  → 학습 데이터의 상위 25% 이상 → high_stress = 1
  → 나머지 → 0
```

> 🔴 **중요:** 이 `1` 은 **임상적 고위험군이 아니다.** 검증된 cut-off가 없어서
> 우리가 연구 편의상 정한 **조작적 정의**다. 분위수를 0.80으로 바꾸면 집단도 바뀐다.
> 결과 문장은 반드시 "조작적으로 정의한 고스트레스 집단"이라고 쓴다.

In [ ]:
# ▶ feature(X) 와 target(y) 의 분류 — 🖐 실행 전에 각자 답해 보게 한 뒤 셀을 돌린다
items = {
    "2015년 가족지지 점수":        "X",
    "2016년 문화적응 스트레스 점수": "y",     # 맞히려는 것은 오직 이것 하나
    "2015년 우울 점수":            "X",
    "2015년 문화적응 스트레스 점수": "X",     # ← 함정! 같은 척도지만 '과거' 정보 = feature (Model B)
}
for k, v in items.items():
    print(f"{v:>3}  {k}")

In [ ]:
# CHECK Step4
try:
    expect = ["X", "y", "X", "X"]
    assert list(items.values()) == expect, f"정답은 {expect}"
    print("✅ PASS — 2015년 스트레스는 target 이 아니라 feature 다 (Model B 전용).")
except Exception as e:
    print("❌ FAIL —", e,
          "\n힌트: 맞히려는 것은 오직 '2016년' 스트레스 하나다. 2015년 것은 과거 정보 = feature.")

<details><summary>💡 해설 (펼쳐 보기)</summary>

```python
items = {
    "2015년 가족지지 점수":        "X",
    "2016년 문화적응 스트레스 점수": "y",
    "2015년 우울 점수":            "X",
    "2015년 문화적응 스트레스 점수": "X",   # Model B 에서만 쓰는 feature
}
```

**핵심:** 같은 이름의 척도라도 **연도가 다르면 역할이 다르다.**
- 2016년 스트레스 = **target** (맞히려는 것)
- 2015년 스트레스 = **feature** (과거 정보)

그래서 우리는 두 모델을 돌린다:
- **Model A** = 2015 심리사회 변인만 → "이전 스트레스를 모를 때 얼마나 구분되나"
- **Model B** = Model A + 2015 스트레스 → "알면 얼마나 나아지나"
</details>

## Step 5 — 시간 설계: 왜 굳이 1년을 건너뛰나 ⏱

가장 쉬운 설계는 **2016년 변수로 2016년 스트레스를 맞히는 것**이다. 성능도 훨씬 잘 나온다.
그런데 우리는 일부러 그렇게 하지 않는다.

```
❌ 쉬운 설계:  2016 우울 · 2016 지지  →  2016 스트레스
   (같은 시점 → "예측"이 아니라 "동시 측정된 것들의 상관")

✅ 우리 설계:  2015 우울 · 2015 지지  →  2016 스트레스
   (1년 뒤를 미리 구분 → 진짜 '예측')
```

같은 시점 데이터로 맞히면 성능은 올라가지만 **아무것도 예측한 게 아니다.**
"오늘 우산을 든 사람을 보고 오늘 비가 왔는지 맞히기"와 같다.

> 이 원칙이 깨지는 사고를 **데이터 누출(data leakage)** 이라 부른다.
> 4차시에서 일부러 누출된 모델을 만들어 AUC가 1.0에 가까워지는 걸 직접 보게 된다.

## Step 6 — 데이터가 실제로 있는지 확인한다 🔍

> **왜 이걸 오늘 하나:** 분석가가 가장 많이 하는 실수는 "데이터가 있겠지" 하고
> 분석 설계부터 짜는 것이다. **존재 → 형식 → 내용**을 눈으로 본 뒤에만 다음으로 간다.

MAPS는 **신청제 자료**다. 웹에서 바로 받을 수 없고, 신청서를 이메일로 보내
승인받아야 한다(약 1주). 이 프로젝트는 그 절차를 거쳐 **2026-08-10 원자료를 수령**했고
`data/raw/` 에 풀어 두었다. 이제 "정말 있는지"를 스크립트로 직접 확인한다.

> 데이터가 없는 환경(새로 clone 한 repo 등)에서 실행하면 🔴 원자료 없음이 뜬다.
> 그것도 오류가 아니라 **결과**다 — 없다는 사실을 정확히 보고하는 것.
> 없는 데이터로 분석을 시작하지 않는다.

In [ ]:
# 우리가 만든 인벤토리 스크립트를 그대로 돌린다.
!python scripts/inspect_raw_data.py

In [ ]:
# 생성된 보고서를 읽어 본다
print(open("reports/data_inventory.md", encoding="utf-8").read()[:1800])

**출력에서 확인할 것** — 존재(데이터 파일 68개) → 형식(CSV·SPSS·STATA 3종) →
내용(매 차수 1,635행 · 차수별 컬럼 수). 그리고 **코드북(xlsx)·조사표(PDF)·유저가이드(PDF)**
가 함께 있는지 본다 — 코드북이 없으면 2차시를 시작할 수 없다.

수령까지의 절차 (우리는 이미 완료했다 — 상세는 `DATA_ACQUISITION.md`):

- 아카이브: https://www.nypi.re.kr/archive → 데이터 다운로드 → 조사표/데이터/코드북
- 신청서 `패널 유형` 칸에 **1기 패널 5차년도, 6차년도** 명시
- 함께 요청: **유저가이드 · 5차/6차 코드북 · 5차/6차 조사표**
- 제출처: `maps@nypi.re.kr` → 약 1주일

In [ ]:
# 스크립트가 제대로 도는지 형식 데모 파일로 확인한다 (⚠️ MAPS 아님)
!python scripts/inspect_raw_data.py --raw data/demo_format --out reports/_demo_inventory.md

In [ ]:
# 데모 인벤토리에서 '두 해 모두 존재하는 ID' 가 몇 명인지 세어 본다
import pandas as pd
w5 = pd.read_csv("data/demo_format/DEMO_wave5_NOT_MAPS.csv")
w6 = pd.read_csv("data/demo_format/DEMO_wave6_NOT_MAPS.csv")

matched = set(w5["DEMO_ID"]) & set(w6["DEMO_ID"])   # & = 교집합 (| 합집합 · - 차집합)
print("5차:", len(w5), "명 / 6차:", len(w6), "명 / 두 해 모두:", len(matched), "명")

In [ ]:
# CHECK Step6 — 위 셀에서 '네가 만든' matched 를 검사한다
try:
    assert len(matched) == 3, f"두 해 모두 응답한 사람은 3명이어야 하는데 {len(matched)}명이 나왔다"
    assert matched == {1001, 1002, 1003}, f"ID 가 다르다: {sorted(matched)}"
    print("✅ PASS — 5명 + 5명이지만 분석 가능한 건 3명뿐이다.")
    print("   사라진 응답자 = 패널 마모(attrition). 실제 MAPS에서도 똑같은 일이 일어난다.")
except NameError:
    print("❌ FAIL — 위 셀을 먼저 실행하라 (matched 가 아직 만들어지지 않았다).")
except Exception as e:
    print("❌ FAIL —", e, "\n힌트: 파이썬 집합의 교집합은 & 연산자다. |(합집합)·-(차집합)와 헷갈리지 말 것.")

<details><summary>💡 해설 (펼쳐 보기)</summary>

```python
matched = set(w5["DEMO_ID"]) & set(w6["DEMO_ID"])
```

`&` = 교집합, `|` = 합집합, `-` = 차집합.

**이게 왜 중요한가:** 패널조사는 해가 갈수록 응답자가 빠진다(전학·이사·거부).
우리 분석 대상은 **5차와 6차에 모두 응답한 사람**뿐이다.
남은 사람이 떠난 사람과 체계적으로 다르다면(예: 스트레스가 심한 학생이 더 많이 이탈)
결과를 전체 다문화청소년으로 일반화할 수 없다 → **8차시 한계 절에 반드시 적는다.**

**실측(실제 MAPS 1기):** 매 차수 파일에는 전체 패널 **1,635명**이 행으로 들어 있다
(미참여자의 응답 칸은 비어 있다). 참여 플래그 기준 5차 **1,347명** · 6차 **1,329명** ·
두 해 모두 참여 **1,321명** — 데모의 "5+5=3"과 똑같은 일이 실제 규모로 일어난 것이다.
</details>

## Step 7 — 연구윤리: 하지 않을 것을 먼저 정한다

이 프로젝트는 **실제 청소년의 심리 데이터**를 다룬다. 시작 전에 경계를 못 박는다.

**하지 않는 것**

- 정신질환 진단 · 임상적 고위험군 판정
- 실제 학생에 대한 자동 개입 결정
- 인과관계 규명
- 학교 현장에 배포할 위험예측 시스템 구축

**지키는 것**

| 규칙 | 이유 |
|---|---|
| 원자료를 Git에 올리지 않는다 | 양도·대여 금지 조항 (NYPI 이용약관) |
| 개인 식별 시도를 하지 않는다 | 연구 목적 외 사용 금지 |
| "고위험군" 대신 "조작적으로 정의한 고스트레스 집단" | 임상 용어를 빌리면 오해가 생긴다 |
| 성능이 나빠도 그대로 보고한다 | 낮은 성능도 결과다 |
| 출처를 명기한다 | "MAPS 1기 5·6차년도 데이터를 활용한 것임" |

> 🤔 **생각해 볼 것:** 만약 이 모델이 정말 잘 맞았다고 하자. 그 결과를 학교에 주면
> 무슨 일이 생길까? 낙인(labeling)은 그 자체로 위험요인이 될 수 있다.

In [ ]:
# 오늘의 산출물 확인 — 이 세 개가 있으면 1차시 완료다
import os
need = {
    "README.md":            "연구 질문과 설계",
    "DATA_ACQUISITION.md":  "데이터 확보 절차",
    "reports/data_inventory.md": "데이터 존재 여부 보고",
}
for f, why in need.items():
    print(f"  {'✅' if os.path.exists(f) else '❌'} {f:32s} {why}")

## 📝 서술형 — 오늘의 연구 질문을 **내 문장으로** 쓴다

아래 셀을 마크다운으로 바꾸고 직접 채운다. (채점 대상)

1. 내가 예측하려는 것은 무엇인가? (한 문장, 연도를 반드시 포함)
2. 무엇으로 예측하는가? (한 문장)
3. 이 결과로 **말할 수 없는 것** 한 가지는?

In [ ]:
# ▶ 서술형 과제 — 여기만은 '내 문장'으로 쓴다 (2차시 시작 때 발표)
# 아래는 채워진 **예시**다. 그대로 베끼지 말고, 각자의 표현으로 다시 쓴다.
예시 = """
1. 나는 (2016)년 (중3) 시점의 (조작적으로 정의한 고문화적응스트레스 집단 소속 여부)를 예측하려 한다.
2. 그 예측에는 (2015)년 시점의 (자아존중감·우울·가족지지 등 심리사회 지표 18개)를 사용한다.
3. 이 결과로 말할 수 없는 것: (어떤 변수가 스트레스의 '원인'이라는 것 — 우리는 연관만 본다)
"""
print(예시)

my_rq = """
1. 나는 (           )년 (       ) 시점의 (                    )를 예측하려 한다.
2. 그 예측에는 (           )년 시점의 (                    )를 사용한다.
3. 이 결과로 말할 수 없는 것: (                                        )
"""
print(my_rq)

## 💾 다음 차시를 위해 — 드라이브에 저장

오늘 만든 것 중 **다음 차시가 재료로 쓰는 파일**을 내 드라이브(`program5_state/`)에 넣어 둔다.
이렇게 해 두면 런타임이 끊겨도, 다른 컴퓨터에서 열어도 **다음 차시가 그냥 시작된다.**

> 🔴 파생 파일이 들어가는 폴더다 — **개인 계정 안에만** 두고 링크 공유·양도하지 않는다.

In [ ]:
# 1차시 산출물 — 2차시가 '무엇을 확인했는지' 이어받는다
handoff_push([
    "reports/data_inventory.md",
])


## 🎯 회고 (5분)

1. "예측한다"와 "원인을 밝힌다"의 차이를 **친구에게 설명한다면** 어떻게 말하겠는가?
2. 우리가 2015년 스트레스를 feature 로 넣는 Model B는, 왜 별도로 돌려 보는가?
3. 이 수업의 설계(연구 질문·변수 후보·윤리 경계)는 데이터가 오기 **전에** 확정됐다.
   데이터 없이 설계부터 확정하는 것이 왜 중요할까?

## ▶️ 다음 (2차시)
> "오늘은 **무엇을 예측할지**를 정했다. 다음엔 **그 변수가 데이터 어디에 있는지**를 찾는다.
> 코드북을 펴고 `configs/variables.yaml` 의 빈칸을 채운다.
> 이때 규칙은 하나 — **컬럼명을 절대 추측하지 않는다.**